In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error

In [ ]:
random_state = 42
seed = 42

In [ ]:
df = pd.read_csv("hotels.csv")

## 1.1

- Working with a dataset that has 32 columns and 119,390 rows, so we seem to have the right data source.
### The Leak
- reservation_status and reservation_status_date are both the leaks. We are trying to model whether bookings will cancel, so our client knows wich one to overbook against. These two columns leak the outcome. "pd.crosstab(df["reservation_status"], df["is_canceled"])" confirms that, showing that a model fed this column would likely get a high accuracy, and not learning the underlying relationship between the data.
- 
#### Missing Values and what they mean
- country:  missing 488 rows, meaning that the hotel did not account or record the guests' country of origin. Useful data, likely has some signal, but miniscule compared to the entire database (119390/488). The best course if action is to replace NaN with "Unknown". Helps that "Unknown" is not a label currently held by "company".

- agent = 16340 missing. This means that no travel agent was involved. Best course of action would be to add a column "has_agent" and fill with a binary identifier (yes or no). The values held by "agent" may  not provide relevant signaling to the model, unless there are specific agents that have irregular cancellation rates, so it may be worth keeping the column and rendering the missing rows as "None".
  
- company = 112593 missing values. This means no company is paying for the stay. Best course of action would be to add a column "has_company" and fill with a binary identifier (yes or no) and drop the column, as it likely produces no valuable signal and is almost entirely pure (112593/119390 = 0.94). Would not be beneficial to keep the values held by this row, like I did with agent, as the specific companies with irregular cancel rates are likely too small to create a predictive relationship (and I don't want to do weighted random sampling for a rare case)

- children = 4 missing. We can assume this was a misinput or an oversight. Also, declared as a floating point number, but should be a regular integer, since the number of children are a discrete value. Means the # of children attending. Fill with zero, too few to matter.

- adr (average daily rate_ : Looking at the desription of adr, the maximum is 5400 and the minimum is -6.3. There are also 1959 rows with an adr of 0. A negative number should be impossible, and a maximum value is extremely high compared to the median price. My motion is to Drop the rows with the negative adr and extreme adr. For the 1959 rows, I will drop 794 rows out of the 1959. I came to this number by finding a way to determine if an actual stay happened. I did that by comparing the adr, total nights stayed (weekends + week days), number of adults, and number of children + babies. If a row has an adr of 0, no adults, and no children, I reason that these stays are non existent and neglile. For the rows that have an adr of 0, but record that there is 1 or more nights and adults recorded is possible that these rows could be complementary visits or zero cost visit (hotel points, coupons, etc.) so I did not remove these rows.

### Other Cleaning Stuff
- Duplicates: After doing the granular cleaning offered by the 1.1 prompt, a basic step is to track the duplicates across the dataset. No booking ID exists, so I can't tell repeated records from coincidentally identical bookings. Dropped them so the same row can't end up in both train and test, which would inflate accuracy. After cleaning the duplicates, the cancel rate dropped from 37% to 27.5%. This i a big drop off and it is possible that some of the data tracked in this dataset was a logging of bookings made, and the dataset may be a result of customer bookings combined with admin room changes that tracks changes using the same database. I plan to compare a model trained with duplicates and without. I expect these two models will differ in their feature understanding.

### Obvious Leak 

In [ ]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

In [ ]:
df = df.drop(columns=["reservation_status", "reservation_status_date"])
df.shape

In [ ]:
df.isna().sum()             # count of NaN values per column

## country

In [ ]:
df["country"] = df["country"].fillna("Unknown") #remove the values holding NaN in country
print(df["country"].isna().sum()) #488 rows under "country" now hold "Unknown", should return 0

### agent

In [ ]:
df["has_agent"] = df["agent"].notna().astype(int)     # flag FIRST, while the blanks still exist

In [ ]:
df["agent"] = df["agent"].fillna(0).astype(int).astype(str)   # IDs as text labels

In [ ]:
df.loc[df["has_agent"] == 0, "agent"] = "None"

In [ ]:
print(df["has_agent"].value_counts())   # about 16,000 zeros
print(df["agent"].nunique())            # number of distinct agents

### company

In [ ]:
df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["company"])

In [ ]:
print(df["has_company"].value_counts())   # about 94% zeros
print("company" in df.columns)            # False

### children

In [ ]:
df["children"] = df["children"].fillna(0).astype(int)
print(df["children"])

In [ ]:
print(df.isna().sum().sum())   # No NaN values
df.shape

### adr (Average Daily Rate)

In [ ]:
#Tackling strange values in adr
df["adr"].describe()


In [ ]:
# check how many rows hold an xtreme value
print("rows over 500:", (df["adr"] > 500).sum())

print("rows over 1000:", (df["adr"] > 1000).sum())

print("rows zero:", (df["adr"] == 0).sum())

print("rows less than or equal to zero:", (df["adr"] <= 0).sum())

print("rows equal to zero:", (df["adr"] == 0).sum())


In [ ]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
guests = df["adults"] + df["children"].fillna(0) + df["babies"]

In [ ]:
df[df["adr"] == 0]["market_segment"].value_counts()

In [ ]:
bad = (df["adr"] < 0) | (df["adr"] > 1000) | (nights == 0) | (guests == 0)
print("rows to drop:", bad.sum())

In [ ]:
zero = df[df["adr"] == 0]
print("zero adr rows:", len(zero))

In [ ]:
zero_nights = zero[(zero["stays_in_weekend_nights"] + zero["stays_in_week_nights"]) == 0]
print("zero adr AND zero nights:", len(zero_nights))

In [ ]:
no_guests = zero[(zero["adults"] + zero["children"].fillna(0) + zero["babies"]) == 0]
print("zero adr AND no guests:", len(no_guests))

In [ ]:
has_no_nights = (zero["stays_in_weekend_nights"] + zero["stays_in_week_nights"]) == 0
has_no_guests = (zero["adults"] + zero["children"].fillna(0) + zero["babies"]) == 0

In [ ]:
to_remove = zero[has_no_nights | has_no_guests]   # | "either one"
remaining = zero[~(has_no_nights | has_no_guests)]  # ~  "not"

In [ ]:
print("removed:  ", len(to_remove))
print("remaining:", len(remaining))
print("check:    ", len(to_remove) + len(remaining))   # should be 1959

In [ ]:
#drop the useless rows.
df = df[~bad].copy()

In [ ]:
#create a new column to track which stays have adr = 0
df["adr_zero"] = (df["adr"] == 0).astype(int)

In [ ]:
print(df.shape)             
print(df["adr_zero"].sum()) 

In [ ]:
before = pd.read_csv("hotels.csv", low_memory=False)["is_canceled"].mean()
print("cancel rate before:", round(before, 4))
print("cancel rate after: ", round(df["is_canceled"].mean(), 4))


### Duplicates

In [ ]:

print("shape before:", df.shape)
print("exact duplicate rows:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
print(df.shape)
print("duplicates left:", df.duplicated().sum())   # should be 0
print("is_canceled rate:", round(df["is_canceled"].mean(), 4))

### Unique value investigation, more leaks?

In [ ]:
for m in ["Undefined", "SC"]:
    sub = df[df["meal"] == m]
    print(m, "| rows:", len(sub), "| cancel rate:", round(sub["is_canceled"].mean(), 4))

In [ ]:
# parking: do canceled bookings ever have parking > 0?
pd.crosstab(df["required_car_parking_spaces"] > 0, df["is_canceled"])


In [ ]:
# room: how often does assigned differ from reserved, split by outcome?
df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"])
print(df.groupby("room_changed")["is_canceled"].agg(["mean", "count"]))

In [ ]:
df = df.drop(columns=["required_car_parking_spaces", "assigned_room_type", "room_changed"])
print(df.shape)
print([c for c in ["required_car_parking_spaces", "assigned_room_type", "room_changed"] if c in df.columns])   # should bef []

### 1.1 Summary of problems found

| Problem | What I did | Why |
|---|---|---|
| **Leaks:** `reservation_status`, `reservation_status_date`, `required_car_parking_spaces`, `assigned_room_type` | Dropped all four | Each records the outcome or happens at check-in. Parking is never above 0 on a canceled booking (0 of 7,290), and a changed room cancels 4.8% vs 31.4%. Not known at booking time. |
| **`agent` missing** (16,340) | Added `has_agent`; kept ID as text with `"None"` | Blank means no travel agent, not unknown. Cancel rate differs (13% without one vs 30% with). |
| **`company` missing** (112,593, 94%) | Added `has_company`; dropped the ID | Blank means no company is paying. IDs are too rare to learn from. |
| **`country` (488) and `children` (4) missing** | `"Unknown"` and 0 | Tiny counts, likely data-entry gaps. Keeps the rows. |
| **Bad `adr`:** negative (1), 5,400 (1), zeros (1,959) | Dropped the 2 impossible rows; kept remaining zeros with an `adr_zero` flag | Impossible or extreme prices would distort a model. Zeros can be real complimentary stays, and they cancel less (13.9%), so the flag lets the model use it. |
| **Bookings with no stay:** 715 zero-night, 180 no-guest | Dropped 825 rows | No actual stay happened. |
| **Exact duplicates** (32,189, 27% of rows) | Dropped them | No booking ID, so I can't tell repeats from coincidences. Dropping avoids the same row landing in both train and test. Cancel rate moved 37.0% to 27.5%. |
| **`meal` `Undefined`** (477 rows) | Kept as its own category | Cancels 15.5% vs 35.6% for `SC`, so merging would hide a real difference. |